# Xếp Hạng CV–Job Tiếng Việt: Đánh Giá Độc Lập và Học Từ Giám Sát Yếu
## Unbiased Evaluation and Weakly Supervised Learning-to-Rank for Vietnamese CV-Job Matching

Thực nghiệm này kế thừa bộ đặc trưng tiên phong từ **Huynh et al. (2025, DEFI)** và mở rộng phương pháp luận theo các nguyên tắc học thuật nghiêm ngặt:
1. **Giải quyết Circular Evaluation**: Đánh giá độc lập trên tập **Job-Disjoint Gold Set** (Dữ liệu Kaggle `JOB_DATA_FINAL.csv` & `USER_DATA_FINAL.csv`).
2. **Dawid-Skene EM Generative Model**: Ước lượng xác suất nhãn yếu $\tilde{y}$ qua thuật toán Dawid-Skene Expectation-Maximization.
3. **Learning-to-Rank (LTR)**: Đề xuất hàm mất mát **Soft-RankNet** chống nhiễu từ giám sát yếu.
4. **Kiểm định Thống kê Nghiêm ngặt**: Paired Bootstrap Resampling ($B=1,000$, lấy mẫu cấp Job) với hiệu chỉnh đa giả thuyết **Holm–Bonferroni** và chỉ số **Circularity Divergence $\mathcal{D}_{circ}$**.

### Bước 1: Khởi Tạo Môi Trường và Tải Dữ Liệu Kaggle (Job-Disjoint Partitioning)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Random seed setup for controlled experiments
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

from src.data_loader import CVJobDatasetLoader, load_dataset, FEATURE_COLS
from src.weak_supervision import WeakSupervisionFramework
from src.models import (
    ModelH_Heuristic,
    ModelA_FixedBCE,
    ModelB_LearnedBCE,
    ModelB_Plus_SoftBCE,
    ModelC_FixedRankNet,
    ModelD_LearnedRankNet,
    ModelD_Plus_ProposedSoftRankNet
)
from src.evaluation import (
    evaluate_models_on_dataset,
    paired_bootstrap_test,
    circularity_divergence,
    apply_holm_bonferroni_correction
)

# Load Real Kaggle Dataset (14,634 Jobs, 3,983 Candidates in data/)
loader_obj = CVJobDatasetLoader(random_seed=42)
df_raw = load_dataset(data_dir='data', random_seed=42)
df_train, df_dev, df_test = loader_obj.get_job_disjoint_splits(df_raw)

print(f"Dataset Partitioning (Job-Disjoint):\n" 
      f"- Train Set (Weak Supervision) : {len(df_train)} pairs across {df_train['job_id'].nunique()} Jobs\n" 
      f"- Dev Set (Gold Validation)    : {len(df_dev)} pairs across {df_dev['job_id'].nunique()} Jobs\n" 
      f"- Test Set (Gold Evaluation)   : {len(df_test)} pairs across {df_test['job_id'].nunique()} Jobs")
df_train.head()

### Bước 2: Dawid-Skene EM Label Model & Kiểm Tra Chất Lượng Nhãn Yếu

In [ ]:
# Initialize Weak Supervision Framework
ws = WeakSupervisionFramework(sem_pct=70, skill_pct=70, exp_pct=60)
df_train_weak = ws.generate_weak_signals(df_train)
df_dev_weak = ws.generate_weak_signals(df_dev)

# 1. Inter-source Agreement
kappa = ws.compute_fleiss_kappa(df_train_weak)
print(f"Inter-Source Agreement (Fleiss' Kappa): {kappa:.4f}")

# 2. Precision on Dev-Gold
dev_precisions = ws.evaluate_precision_on_dev(df_dev_weak)
print("Weak Signal Precision on Dev-Gold Human Labels:")
for lf_name, prec in dev_precisions.items():
    print(f"  - {lf_name}: {prec*100:.2f}%")

# 3. Fit Dawid-Skene EM Model
df_train_weak = ws.predict_probabilistic_labels(df_train_weak)
print(f"Dawid-Skene Sensitivity (alpha) : {ws.source_sensitivities}")
print(f"Dawid-Skene Specificity (beta)  : {ws.source_specificities}")
df_train_weak[['job_id', 'cand_id', 'lf_sem', 'lf_skill', 'lf_exp', 'y_prob']].head(10)

### Bước 3: Huấn Luyện Các Mô Hình Trong Ma Trận Ablation (H $\rightarrow$ D+)

In [ ]:
models = {
    'H (Heuristic)': ModelH_Heuristic(),
    'A (Baseline BCE)': ModelA_FixedBCE(),
    'B (Learned BCE)': ModelB_LearnedBCE(),
    'B+ (Soft BCE)': ModelB_Plus_SoftBCE(),
    'C (Fixed RankNet)': ModelC_FixedRankNet(epochs=30),
    'D (Main RankNet)': ModelD_LearnedRankNet(epochs=40),
    'D+ (Proposed Soft-RankNet)': ModelD_Plus_ProposedSoftRankNet(epochs=50)
}

print("Training models...")
for name, m in models.items():
    if hasattr(m, 'fit'):
        m.fit(df_train_weak)
        print(f" -> Model '{name}' trained successfully.")
    else:
        print(f" -> Model '{name}' ready (no training needed).")

### Bước 4: Đánh Giá Hiệu Năng Độc Lập trên Gold Set vs Heuristic Test Set

In [ ]:
res_gold = evaluate_models_on_dataset(models, df_test, target_col='gold_relevance')
res_heur = evaluate_models_on_dataset(models, df_test, target_col='heuristic_score')

print("=== BẢNG 1: HIỆU NĂNG XẾP HẠNG TRÊN GOLD SET ĐỘC LẬP (JOB-DISJOINT) ===")
display(res_gold)

print("\n=== BẢNG 2: HIỆU NĂNG TRÊN TẬP TEST HEURISTIC (CIRCULAR EVALUATION) ===")
display(res_heur)

### Bước 5: Paired Bootstrap Resampling (B=1,000) & Hiệu Chỉnh Holm-Bonferroni

In [ ]:
# Compute raw bootstrap tests
raw_tests = {
    'H1 (Weight Learning : B vs A)': paired_bootstrap_test(df_test, models['A (Baseline BCE)'], models['B (Learned BCE)'], k=10),
    'H2 (Pairwise LTR    : D vs B)': paired_bootstrap_test(df_test, models['B (Learned BCE)'], models['D (Main RankNet)'], k=10),
    'Proposed Soft-RankNet: D+ vs A': paired_bootstrap_test(df_test, models['A (Baseline BCE)'], models['D+ (Proposed Soft-RankNet)'], k=10)
}

# Apply Holm-Bonferroni correction
corrected_tests = apply_holm_bonferroni_correction(raw_tests, alpha=0.05)

print("=== BẢNG 3: BẰNG CHỨNG KIỂM ĐỊNH THỐNG KÊ (HOLM-BONFERRONI ADJUSTED) ===")
for test_name, res in corrected_tests.items():
    sig_str = "(Significant - Holm-Bonferroni)" if res['is_significant_holm'] else "(Not Significant)"
    print(f"{test_name:32s} -> Delta nDCG@10 = {res['mean_delta']:+.4f} [95% CI: {res['ci_95_low']:.4f}, {res['ci_95_high']:.4f}], p = {res['p_value']:.4f} {sig_str}")

# Hypothesis H3: Circularity Divergence Index
circ_info = circularity_divergence(res_heur, res_gold, metric_name='nDCG@10')
print("\n=== BẢNG 4: CHỈ SỐ SAI LỆCH THỨ HẠNG CIRCULARITY DIVERGENCE (H3) ===")
print(f"Kendall's Tau_b (Heuristic vs Gold): {circ_info['tau_b']:.4f} (p = {circ_info['kendall_p_val']:.4f})")
print(f"Circularity Divergence Index (D_circ): {circ_info['d_circ']:.4f}")
print(f"Rank Reversal Confirmed (D_circ > 0.3): {circ_info['has_rank_reversal']}")

### Bước 6: Trực Quan Hóa Kết Quả Phục Vụ Bài Báo (Publication-Ready Figures)

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=res_gold, x='Model', y='nDCG@10', hue='Model', legend=False, palette='Blues_d')
plt.title('Performance Comparison on Independent Gold Set (nDCG@10)', fontsize=14, fontweight='bold')
plt.xticks(rotation=25)
plt.ylim(0, 1.15)
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### Hạn Chế và Hướng Phát Triển (Limitations & Threats to Validity)

1. **Cỡ mẫu Gold Set**: Tập Gold Set gồm 12 Jobs thử nghiệm đại diện cho miền dữ liệu tuyển dụng Kaggle Việt Nam. Việc mở rộng quy mô gán nhãn đa miền (Finance, Tech, Sales) là hướng đi cần thiết.
2. **Đặc trưng Tiếng Việt sâu hơn**: Hiện tại sử dụng TF-IDF và trích xuất chuẩn hóa. Việc thay thế bằng PhoBERT / ViSBERT fine-tuned với Contrastive Loss sẽ tăng cường độ mịn ngữ nghĩa cho `desc_sem_sim`.
3. **Hiệu chỉnh đa giả thuyết**: Đã áp dụng Holm–Bonferroni để đảm bảo FWER $< 0.05$ trong toàn bộ kiểm định bootstrap.